# Polish Forms fine-tuning: Qwen3VL LoRA + TrOCR

Upload `polish_forms_finetuning_colab_bundle.zip` to Google Drive, then run the cells below. The notebook assumes the zip is in `MyDrive/Magisterka/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi
!python --version

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/Magisterka/polish_forms_finetuning_colab_bundle.zip'
!rm -rf /content/FineTuning /content/ocr_benchmark_utils.py
!unzip -q "$ZIP_PATH" -d /content
!find /content/FineTuning -maxdepth 2 -type f | head -30

In [ ]:
%cd /content
!pip -q install -r /content/FineTuning/requirements_colab.txt

In [ ]:
import pandas as pd, json
root = '/content/FineTuning/data'
for name in ['train', 'val', 'test', 'train_dysgraphia_oversampled']:
    df = pd.read_csv(f'{root}/manifests/{name}.csv')
    print(name, len(df), df['difficulty_group'].value_counts().to_dict())
print(json.load(open(f'{root}/dataset_summary.json', encoding='utf-8'))['balanced_train_counts_by_group'])

## Smoke tests

These only check that processors and batches load correctly. They do not train.

In [ ]:
!python /content/FineTuning/TrOCR/train_trocr.py --train-limit 2 --val-limit 2 --dry-run
!python /content/FineTuning/Qwen3VL_LoRA/train_qwen3vl_lora.py --train-limit 2 --val-limit 2 --dry-run

## Qwen3VL LoRA: standard

This is the primary experiment. On a strong GPU start with 3 epochs. If memory is tight, keep `--batch-size 1`, `--grad-accum 8`, and `--gradient-checkpointing`.

In [ ]:
!python /content/FineTuning/Qwen3VL_LoRA/train_qwen3vl_lora.py \
  --train-json /content/FineTuning/data/qwen/train.json \
  --val-json /content/FineTuning/data/qwen/val.json \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/outputs/qwen3vl_lora_standard \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8 \
  --gradient-checkpointing

In [ ]:
!python /content/FineTuning/Qwen3VL_LoRA/evaluate_qwen3vl_lora.py \
  --adapter-path /content/drive/MyDrive/Magisterka/outputs/qwen3vl_lora_standard \
  --test-json /content/FineTuning/data/qwen/test.json \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/evaluation/qwen3vl_lora_standard \
  --run-name qwen3vl_lora_standard

## Qwen3VL LoRA: dysgraphia oversampling

In [ ]:
!python /content/FineTuning/Qwen3VL_LoRA/train_qwen3vl_lora.py \
  --train-json /content/FineTuning/data/qwen/train_dysgraphia_oversampled.json \
  --val-json /content/FineTuning/data/qwen/val.json \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/outputs/qwen3vl_lora_dysgraphia_oversampled \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8 \
  --gradient-checkpointing

In [ ]:
!python /content/FineTuning/Qwen3VL_LoRA/evaluate_qwen3vl_lora.py \
  --adapter-path /content/drive/MyDrive/Magisterka/outputs/qwen3vl_lora_dysgraphia_oversampled \
  --test-json /content/FineTuning/data/qwen/test.json \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/evaluation/qwen3vl_lora_dysgraphia_oversampled \
  --run-name qwen3vl_lora_dysgraphia_oversampled

## TrOCR-large baseline fine-tuning

Run after Qwen, or in a fresh runtime.

In [ ]:
!python /content/FineTuning/TrOCR/train_trocr.py \
  --model-id microsoft/trocr-large-handwritten \
  --train-csv /content/FineTuning/data/trocr/train.csv \
  --val-csv /content/FineTuning/data/trocr/val.csv \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/outputs/trocr_large_standard \
  --epochs 10 \
  --batch-size 2 \
  --grad-accum 8 \
  --gradient-checkpointing

In [ ]:
!python /content/FineTuning/TrOCR/evaluate_trocr.py \
  --model-path /content/drive/MyDrive/Magisterka/outputs/trocr_large_standard \
  --test-csv /content/FineTuning/data/trocr/test.csv \
  --data-root /content/FineTuning/data \
  --output-dir /content/drive/MyDrive/Magisterka/evaluation/trocr_large_standard \
  --run-name trocr_large_standard

In [ ]:
import pandas as pd, glob
for path in glob.glob('/content/drive/MyDrive/Magisterka/evaluation/*/*_summary.csv'):
    print('\n', path)
    display(pd.read_csv(path)[['model_id', 'samples', 'corpus_cer', 'corpus_wer', 'corpus_cla', 'corpus_crw']])